# Global Sensitivity B — n_prior_periods Strategy-specific Tuning

공식 A와 Step 2 B locked 결과는 읽기만 한다. 이 Notebook은 Global Train에서 확정 25개 Feature와 기존 `n_prior_periods`를 결합한 26개 Feature로만 제한 GridSearch와 Train OOF를 실행한다. Test·Feature Selection·threshold tuning은 사용하지 않는다.


## 1. 입력·Feature 계약·예상 학습량


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

ROOT = Path(os.environ.get("KHUDA_PROJECT_ROOT", Path.cwd())).resolve()
while not (ROOT / "code").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("저장소 안에서 Notebook을 실행하거나 KHUDA_PROJECT_ROOT를 지정하세요.")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
    del sys.modules["code"]

from code.contracts import DatasetBundle
from code.evaluation.evaluate import calculate_binary_metrics
from code.model.locked_sensitivity import load_stage_3_5_locked_params
from code.model.strategy_tuning import (
    count_grid_combinations,
    run_b_lr_tuning,
    run_c_lr_tuning,
    run_xgb_tuning_stage_a,
    run_xgb_tuning_stage_b,
    run_xgb_tuning_stage_c,
    save_strategy_tuning_artifacts,
    step3_tuning_grids,
)
from code.pipeline.audit import (
    attach_person_period_column,
    calculate_train_sample_weight,
    load_selected_feature_names,
)
from code.pipeline.saved_results import load_saved_global_train, select_bundle_features

RESULT_ROOT = ROOT / "data" / "result" / "baseline_42features"
DATASET_PATH = RESULT_ROOT / "datasets" / "global_dataset.parquet"
SPLIT_PATH = RESULT_ROOT / "splits" / "split_ids.csv"
FEATURE_CONFIG = ROOT / "code" / "config" / "features.yaml"
MODEL_CONFIG = ROOT / "code" / "config" / "model_config.yaml"
STAGE35 = RESULT_ROOT / "modeling" / "stage_3_5"
SELECTED = RESULT_ROOT / "modeling" / "stage_3" / "selected_features.csv"
PARAMS = STAGE35 / "final_refined_params.json"
A_SUMMARY = STAGE35 / "final_tuning_summary.csv"
A_FOLD = STAGE35 / "final_fold_f1.json"
A_OOF = {model: STAGE35 / f"refined_{model}_oof_predictions.parquet" for model in ("logistic_regression", "xgboost")}
# Jupyter worker의 표준 라이브러리 code 이름 충돌을 피한다. GridSearch 병렬 수는 n_jobs=-1을 유지한다.
PARALLEL_BACKEND = "threading"
COLORS = {"A Baseline":"#7a7a7a", "Locked":"#0066cc", "Tuned":"#2997ff"}

PERSON_PERIOD_PATH = RESULT_ROOT / "datasets" / "person_period.parquet"
LOCKED_DIR = RESULT_ROOT / "modeling" / "sensitivity_n_prior_periods"
OUT = LOCKED_DIR / "tuning"
LOCKED_SUMMARY = LOCKED_DIR / "locked_model_summary.csv"
LOCKED_FOLD = LOCKED_DIR / "locked_fold_f1.json"
LOCKED_OOF = {model: LOCKED_DIR / f"locked_{model}_oof_predictions.parquet" for model in ("logistic_regression", "xgboost")}


In [ ]:
required = [DATASET_PATH, SPLIT_PATH, PERSON_PERIOD_PATH, FEATURE_CONFIG, MODEL_CONFIG, SELECTED, PARAMS, A_SUMMARY, A_FOLD, LOCKED_SUMMARY, LOCKED_FOLD, *A_OOF.values(), *LOCKED_OOF.values()]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("필요한 저장 산출물이 없습니다:\n" + "\n".join(missing))
base = load_saved_global_train(DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG)
selected = load_selected_feature_names(SELECTED)
train_frame = base.to_frame()
train_with_prior = attach_person_period_column(train_frame, PERSON_PERIOD_PATH, "n_prior_periods")
TRAIN = DatasetBundle(
    name="global_step3_b",
    X=train_with_prior.loc[:, [*selected, "n_prior_periods"]].reset_index(drop=True),
    y=base.y.copy(),
    groups=base.groups.copy(),
    metadata=base.metadata.copy(),
    sample_weight=base.sample_weight.copy(),
)
params = load_stage_3_5_locked_params(PARAMS)
grids = step3_tuning_grids(MODEL_CONFIG, strategy="B")
assert TRAIN.X.shape[1] == 26
assert TRAIN.X.columns.tolist() == [*selected, "n_prior_periods"]
display(pd.DataFrame([{
    "strategy": "B tuned", "train_rows": len(TRAIN.y), "unique_SAMPID": TRAIN.groups.nunique(),
    "feature_count": TRAIN.X.shape[1], "sample_weight_used": False, "test_used": False,
    "lr_combinations": count_grid_combinations(grids["logistic_regression"]), "lr_expected_fits": 140,
    "xgb_expected_fits": 585, "core_cv_fits": 725,
}]))


## 2. 제한 GridSearch — B tuned

실행 시 LR 28개 조합과 XGBoost Phase A → B → C 제한 Grid만 실행한다. 범위는 자동 확장하지 않는다.


In [ ]:
lr_tuned = run_b_lr_tuning(TRAIN, feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG, parallel_backend_name=PARALLEL_BACKEND, n_jobs=-1)
xgb_a = run_xgb_tuning_stage_a(TRAIN, strategy="B", stage35_params=params["xgboost"], sample_weight_train=None, feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG, parallel_backend_name=PARALLEL_BACKEND, n_jobs=-1)
xgb_b = run_xgb_tuning_stage_b(TRAIN, strategy="B", stage_a_params=xgb_a.best_params, sample_weight_train=None, feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG, parallel_backend_name=PARALLEL_BACKEND, n_jobs=-1)
xgb_tuned = run_xgb_tuning_stage_c(TRAIN, strategy="B", stage_b_params=xgb_b.best_params, sample_weight_train=None, feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG, parallel_backend_name=PARALLEL_BACKEND, n_jobs=-1)
tuned_results = [lr_tuned, xgb_tuned]
fold_results = pd.concat([
    pd.DataFrame({"fold": range(5), "strategy": item.strategy, "model": item.search.model, "f1": item.search.fold_f1})
    for item in tuned_results
], ignore_index=True)
display(pd.DataFrame([item.summary_row() for item in tuned_results]))
display(fold_results)


## 3. B tuned 전용 저장

Step 2 B locked 및 A 공식 결과는 덮어쓰지 않는다.


In [ ]:
artifact_paths = save_strategy_tuning_artifacts(output_dir=OUT, bundle=TRAIN, tuned_results=tuned_results, lr_search=lr_tuned.search, xgb_stage_a=xgb_a, xgb_stage_b=xgb_b, xgb_stage_c=xgb_tuned.search)
display(pd.DataFrame({"artifact": artifact_paths.keys(), "path": [str(path) for path in artifact_paths.values()]}))


## 4. A · B locked · B tuned 비교

자동 해석이나 Strategy 선택은 생성하지 않는다.


In [ ]:
def normalize_oof(frame):
    return frame.rename(columns={
        "y_proba": "y_probability",
        "y_pred": "y_predicted",
        "y_pred_at_0_5": "y_predicted",
    })

def rows_from_saved(strategy, summary_path, oof_paths, *, stage=None, parameter_status="locked"):
    summary = pd.read_csv(summary_path)
    if stage is not None:
        summary = summary.loc[summary["stage"].eq(stage)].copy()
    summary = summary.set_index("model")
    if not summary.index.is_unique:
        raise ValueError(f"{strategy} summary에는 모델별 한 행만 있어야 합니다.")
    rows, oofs = [], {}
    for model, path in oof_paths.items():
        oof = normalize_oof(pd.read_parquet(path))
        metric = calculate_binary_metrics(oof["y_true"], oof["y_probability"])
        oofs[model] = oof
        rows.append({
            "strategy": strategy,
            "model": model,
            "feature_count": int(summary.loc[model, "feature_count"]),
            "parameter_status": parameter_status,
            "cv_f1_mean": summary.loc[model, "cv_f1_mean"],
            "cv_f1_std": summary.loc[model, "cv_f1_std"],
            "oof_precision": metric["precision"],
            "oof_recall": metric["recall"],
            "oof_f1": metric["f1"],
            "oof_roc_auc": metric["roc_auc"],
            "oof_average_precision": metric["average_precision"],
            "predicted_positive_rate": oof["y_predicted"].mean(),
            "best_params": summary.loc[model, "best_params"] if "best_params" in summary else None,
        })
    return pd.DataFrame(rows), oofs

def tuned_rows(results):
    rows, oofs = [], {}
    for result in results:
        row = result.summary_row()
        row["feature_count"] = TRAIN.X.shape[1]
        row["parameter_status"] = "tuned"
        rows.append(row)
        oofs[result.search.model] = result.oof_predictions
    return pd.DataFrame(rows), oofs

def add_f1_deltas(comparison, locked_label, tuned_label):
    baseline = comparison.query("strategy == 'A Baseline'").set_index("model")["oof_f1"]
    locked = comparison.query("strategy == @locked_label").set_index("model")["oof_f1"]
    result = comparison.copy()
    result[f"{locked_label}_minus_A_oof_f1"] = result.apply(lambda row: row.oof_f1 - baseline[row.model], axis=1)
    result[f"{tuned_label}_minus_A_oof_f1"] = result.apply(lambda row: row.oof_f1 - baseline[row.model], axis=1)
    result[f"{tuned_label}_minus_{locked_label}_oof_f1"] = result.apply(lambda row: row.oof_f1 - locked[row.model], axis=1)
    return result

a_rows, _ = rows_from_saved("A Baseline", A_SUMMARY, A_OOF, stage="stage_3_5")
locked_rows, locked_oof = rows_from_saved("B locked", LOCKED_SUMMARY, LOCKED_OOF)
locked_rows["best_params"] = locked_rows["model"].map(params)
tuned_rows_frame, tuned_oof = tuned_rows(tuned_results)
comparison = pd.concat([a_rows, locked_rows, tuned_rows_frame], ignore_index=True)
comparison = add_f1_deltas(comparison, "B locked", "B tuned")
display(comparison)
with LOCKED_FOLD.open(encoding="utf-8") as file:
    locked_fold = json.load(file)


## 5. 비교 그래프


In [ ]:
def plot_comparison(comparison, saved_oof, tuned_oof, saved_fold, tuned_results, *, locked_label, tuned_label, include_ppr):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        rows = comparison.query("model == @model")
        colors = [COLORS["A Baseline"], COLORS["Locked"], COLORS["Tuned"]]
        ax.bar(rows.strategy, rows.cv_f1_mean, yerr=rows.cv_f1_std, color=colors, capsize=4)
        ax.set_title(f"{model}: CV F1 mean ± std")
    plt.tight_layout(); plt.show()

    with A_FOLD.open(encoding="utf-8") as file:
        baseline_fold = json.load(file)["stage_3_5"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        ax.plot(range(1, 6), baseline_fold[model], marker="o", label="A Baseline", color=COLORS["A Baseline"])
        ax.plot(range(1, 6), saved_fold[model], marker="o", label=locked_label, color=COLORS["Locked"])
        ax.plot(range(1, 6), next(item.search.fold_f1 for item in tuned_results if item.search.model == model), marker="o", label=tuned_label, color=COLORS["Tuned"])
        ax.set_title(f"{model}: 5-fold F1"); ax.set_xticks(range(1, 6)); ax.legend()
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    metrics = ["oof_precision", "oof_recall", "oof_f1"]
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        rows = comparison.query("model == @model").set_index("strategy").loc[["A Baseline", locked_label, tuned_label]]
        positions = np.arange(len(metrics)); width = 0.25
        for offset, (label, row, color) in enumerate(zip(rows.index, rows.itertuples(), [COLORS["A Baseline"], COLORS["Locked"], COLORS["Tuned"]])):
            ax.bar(positions + (offset - 1) * width, [getattr(row, metric) for metric in metrics], width, label=label, color=color)
        ax.set_xticks(positions, ["Precision", "Recall", "F1"]); ax.set_title(f"{model}: OOF metrics"); ax.legend()
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        for label, collection, color in [("A Baseline", A_OOF, COLORS["A Baseline"]), (locked_label, saved_oof, COLORS["Locked"]), (tuned_label, tuned_oof, COLORS["Tuned"])]:
            value = collection[model]
            oof = normalize_oof(pd.read_parquet(value)) if isinstance(value, (str, Path)) else value
            RocCurveDisplay.from_predictions(oof["y_true"], oof["y_probability"], name=label, ax=ax, curve_kwargs={"color": color})
        ax.set_title(f"{model}: ROC")
    plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        for label, collection, color in [("A Baseline", A_OOF, COLORS["A Baseline"]), (locked_label, saved_oof, COLORS["Locked"]), (tuned_label, tuned_oof, COLORS["Tuned"])]:
            value = collection[model]
            oof = normalize_oof(pd.read_parquet(value)) if isinstance(value, (str, Path)) else value
            PrecisionRecallDisplay.from_predictions(oof["y_true"], oof["y_probability"], name=label, ax=ax, curve_kwargs={"color": color})
        ax.set_title(f"{model}: Precision-Recall")
    plt.tight_layout(); plt.show()

    if include_ppr:
        fig, ax = plt.subplots(figsize=(7, 4))
        rows = comparison.query("model == 'xgboost'")
        ax.bar(rows.strategy, rows.predicted_positive_rate, color=[COLORS["A Baseline"], COLORS["Locked"], COLORS["Tuned"]])
        ax.set_title("XGBoost: predicted positive rate")
        plt.tight_layout(); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, model in zip(axes, ("logistic_regression", "xgboost")):
        rows = comparison.query("model == @model").set_index("strategy")
        ax.bar(["Locked − A", "Tuned − Locked"], [rows.loc[locked_label, "oof_f1"] - rows.loc["A Baseline", "oof_f1"], rows.loc[tuned_label, "oof_f1"] - rows.loc[locked_label, "oof_f1"]], color=[COLORS["Locked"], COLORS["Tuned"]])
        ax.axhline(0, color="#7a7a7a", linewidth=1); ax.set_title(f"{model}: OOF F1 change")
    plt.tight_layout(); plt.show()

plot_comparison(comparison, locked_oof, tuned_oof, locked_fold, tuned_results, locked_label="B locked", tuned_label="B tuned", include_ppr=False)
